In [1]:
%load_ext autoreload
%autoreload 2

# Worm Re-segmentation — Worker Notebook

สมุดนี้สำหรับ **คนทำงาน** แก้ mask หนอนด้วย GUI

**ก่อนรัน**
1. ต้องอยู่บน **GPU node**: `cd /project/lt200264-saiwat/WormProject/code/slurm && sbatch run_jupyter_worm.sh`
2. เลือก kernel **`segmentation`** (ห้าม `Python 3 (ipykernel)`)
3. **แก้ชื่อตัวเอง** ในเซลล์ถัดไปก่อนรันอย่างอื่น

> งานถูกแจกให้แต่ละคนแล้วโดยผู้จัดการ — คุณจะเห็นเฉพาะภาพของตัวเอง ไม่ชนกับคนอื่น

## 1. ตั้งชื่อตัวเอง

แก้ `MY_NAME` เป็นชื่อที่ผู้จัดการแจกงานให้ (ภาษาอังกฤษ ตรงตัว ไม่มีเว้นวรรค)

In [2]:
import sys
sys.path.insert(0, "/project/lt200264-saiwat/WormProject/code")

# ╔══════════════════════════════════════════════════════════════╗
# ║  แก้ตรงนี้ — ใส่ชื่อตัวเอง (ต้องตรงกับที่ผู้จัดการตั้งให้)     ║
# ╚══════════════════════════════════════════════════════════════╝
MY_NAME = "gun"                          # ← ชื่อของคุณ
# TEAM    = ["karn", "thas", "gun"]           # ← รายชื่อทุกคนในทีม (ต้องตรงกันทุกคน)

assert MY_NAME != "changeme", "⚠️ ยังไม่ได้เปลี่ยนชื่อ! แก้ MY_NAME ก่อนรัน"
print(f"✓ สวัสดี {MY_NAME}")

LABEL_CSV  = "/project/lt200264-saiwat/WormProject/data/worm-24022026/labels/worm-24022026-labels_current.csv"
ASSIGN_CSV = "/scratch/lt200264-saiwat/resegmentation_labels_v2/assign.csv"

✓ สวัสดี gun


## 2. เปิด GUI แก้ mask

เริ่มเป็น **dry-run** (กดได้ทุกปุ่มแต่ยังไม่เขียน pkl จริง) — กด 🔓 ในหน้าจอเพื่อเปลี่ยนเป็นเขียนจริง

### วิธีใช้

| ทำอะไร | วิธี |
|---|---|
| บอกว่า "ตรงนี้คือหนอน" | **คลิกซ้าย** บนภาพ |
| บอกว่า "ตรงนี้ไม่ใช่" | **คลิกขวา** หรือ Shift + คลิกซ้าย |
| จำกัดขอบเขต | **ลากเมาส์** เป็นกรอบ |
| ลบจุดที่วางไว้ | คลิกทับจุดเดิม |
| เลือกผลจากโมเดล | กด `1`-`6` (หรือกดปุ่มใต้ภาพย่อ) |
| บันทึกทับ mask เดิม | `S` |
| หนอนตัวที่ระบบมองข้าม (ตัวแถม) | `A` |
| ระบุไม่ได้ / undecidable | `U` — ตั้ง bUse=6 (ถ้าคลิก manual ไว้จะบันทึก mask ใหม่ด้วย ถ้าไม่คลิกก็แค่เปลี่ยน bUse) |
| แถม + ระบุไม่ได้ | `D` (เหมือน `A` แต่ตั้ง bUse=6) |
| Undo | `Z` |
| ล้างจุดทั้งหมด | `X` |
| คำนวณ mask ใหม่ | `R` |
| mask พัง แก้ไม่ได้ | `F` (fail — เก็บไว้กลับมา retry ได้) |
| mask ซ้ำกับตัวอื่น | `C` (duplicate — ตัดออกถาวร) |
| ปิดการระบายสี ให้เหลือแต่เส้นขอบ | `B` |

สไลเดอร์ **🔍 บริบท** ขยายหน้าต่างจากภาพ raw — ใช้ดูว่าหนอนตัวข้างๆ อยู่ตรงไหน

> ทุกภาพต้องกดปุ่มใดปุ่มหนึ่ง (`S` / `A` / `D` / `U` / `F` / `C`) ไม่มีปุ่มข้าม

In [15]:
from tool.worm_reseg_gui2 import launch, load_queue

q = load_queue(LABEL_CSV, class_filter=5, assign=(ASSIGN_CSV, MY_NAME))
print(f"งานของ {MY_NAME}: {len(q):,} แถว จาก {q.stem.nunique():,} ภาพ")

gui = launch(queue=q, dry_run=False)

งานของ gun: 3,146 แถว จาก 202 ภาพ
ℹ️ ข้าม 263 ภาพที่ถูก resegment ไปแล้ว (bUse=3) — เหลือ 2883 รอทำ


## 3. ตรวจผลด้วย Mask Viewer

เปิดดู mask ทั้งหมดในภาพ — เปลี่ยน bUse ได้ / ลบได้เฉพาะตัวแถม (ตัวจริงลบไม่ได้)

- **Filter**: เลือก bUse ที่ต้องการดู (เริ่มที่ 3 = Repaired, เปลี่ยนเป็น -1 เพื่อดูทั้งหมด)
- **Set bUse + Apply**: เปลี่ยนสถานะ mask ที่ติ๊กไว้
- **Delete**: ลบตัวแถมที่ไม่ต้องการ (ต้องกด 2 จังหวะ)
- รายละเอียดเพิ่มเติมดู `worm_mask_viewer_demo.ipynb`

In [14]:
from tool import worm_mask_viewer

viewer = worm_mask_viewer.launch(dry_run=False)

## 4. ดูความคืบหน้าของตัวเอง

In [11]:
import collections
from tool.worm_reseg_gui2 import Session

SAVE = "/scratch/lt200264-saiwat/resegmentation_labels_v2"
s = Session(SAVE, user=MY_NAME)

my_items = {fn: st for fn, (st, by) in s.state.items() if by == MY_NAME}
print(f"=== ความคืบหน้าของ {MY_NAME} ===")
print(collections.Counter(my_items.values()))
print(f"รวม {len(my_items)} รายการ")

=== ความคืบหน้าของ gun ===
Counter()
รวม 0 รายการ


---

## ค่า bUse แต่ละอันคืออะไร

| bUse | ชื่อ | ความหมาย |
|:---:|---|---|
| **0** | Noise | ไม่ใช่หนอน — เศษอาหาร/ขยะ |
| **1** | Kept | ผ่าน filter → ใช้งานได้ |
| **2** | Edge case | กรณีพิเศษจาก pipeline เก่า (หายาก) |
| **3** | Repaired | คนวาด mask ใหม่ด้วย reseg GUI |
| **4** | Failed | ลองแก้แล้วไม่ได้ — กลับมา retry ได้ |
| **5** | Duplicate | ซ้ำกับ mask อื่น — ตัดออกถาวร |
| **6** | Undecidable | ติดขอบภาพ/เห็นไม่ครบ — ตัดสินไม่ได้ |

## Output ไปไหนบ้าง

ทุกอย่างเขียนใต้ `/scratch/lt200264-saiwat/resegmentation_labels_v2/`

| โฟลเดอร์ | เนื้อใน |
|---|---|
| `roi_src/` + `roi_mask/` | ภาพ ROI + mask ใหม่ (คู่ส่งไป relabel) |
| `extra/` | หนอนตัวแถม (กด `A`) |
| `failed/` , `duplicate/` | ตัวที่กด `F` / `C` |
| `pkl_backup/` | สำเนา pkl ก่อนถูกแตะครั้งแรก (กู้คืนได้) |
| `session_state.<ชื่อ>.csv` | ประวัติว่าทำอะไรไปแล้ว |

**pkl หลัก** (production) ถูกเขียนทับที่ `processed/segmentation_masks/<stem>.pkl` ตั้ง `bUse=3`